# 02 — Guardrails

> **ML & AI Engineering Cookbook** — hands-on Jupyter notebook.
>
> Run on [Google Colab](https://colab.research.google.com/) or locally with Jupyter.

## Notebook Learning Structure

1. **Concept Overview** — understand the problem and its fundamentals.
2. **Mathematical Foundation** — explore the underlying mathematics.
3. **Implementation from Scratch** — build it with Python and NumPy.
4. **Library Implementation** — use the industry-standard framework.
5. **Dataset Exploration** — work with a practical dataset.
6. **Visualization** — understand results through plots.
7. **Model Evaluation** — analyze performance with metrics.
8. **Experiments** — modify parameters and observe behavior.
9. **Hands-on Challenges** — practice independently.
10. **Interview Questions** — reinforce conceptual understanding.


---

## 1. Concept Overview — Guardrails for LLMs


Guardrails are runtime checks that prevent an LLM from producing harmful,
off-topic, or out-of-policy outputs. They sit between the user and the
model: input guardrails filter prompts, output guardrails filter responses.

**Beginner track:** learn the three input checks (PII, prompt injection,
topic) and three output checks (toxicity, factuality, format).
**Advanced track:** also learn `guardrails-ai`, NeMo Guardrails, and how
to combine guardrails with structured output constraints.


---

## 2. Mathematical Foundation — Mathematical Foundation


A guardrail is a classifier $g: x \to \{\text{pass}, \text{block}\}$. The
pipeline runs $g_\text{in}$ on the input, then the LLM, then $g_\text{out}$
on the output. If either blocks, the request is rejected.

Each guardrail is itself a model — usually a small classifier (toxicity,
topic) or a regex (PII, format). Composing them is the engineering.


---

## 3. Implementation from Scratch — Implementation from Scratch


**What this does:** Implements four guardrails from scratch: PII detection
(regex for emails and phone numbers), prompt injection detection (keyword
matching), toxicity detection (keyword matching — a real system would use
a classifier), and output format validation (JSON schema).

**Why it matters:** guardrails look like magic until you've written one.
Then you see they're just classifiers (or regexes) composed in a pipeline.

**What to watch for:** regex is a poor PII detector in practice — it misses
many real emails and matches false positives. Use a dedicated PII library
(`presidio`) in production.

In [ ]:
import re
from typing import Optional

# --- Input guardrails ---

def check_pii(text):
    """Block if text contains email or phone-like patterns."""
    email_re = r"\b[\w.-]+@[\w.-]+\.\w+\b"
    phone_re = r"\b\+?\d{1,3}?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3,4}[-.\s]?\d{4}\b"
    if re.search(email_re, text):
        return False, "PII detected: email address"
    if re.search(phone_re, text):
        return False, "PII detected: phone number"
    return True, None

def check_prompt_injection(text):
    """Block if text contains common prompt-injection patterns."""
    patterns = [
        "ignore previous instructions",
        "you are now",
        "disregard the above",
        "forget your rules",
        "system prompt:",
    ]
    t = text.lower()
    for p in patterns:
        if p in t:
            return False, f"prompt injection detected: pattern '{p}'"
    return True, None

# --- Output guardrails ---

TOXIC_WORDS = {"hate", "kill", "stupid", "idiot", "moron"}  # tiny demo list

def check_toxicity(text):
    """Block if text contains toxic words (toy implementation)."""
    tokens = set(text.lower().split())
    found = tokens & TOXIC_WORDS
    if found:
        return False, f"toxic words: {found}"
    return True, None

def check_json_format(text, schema_keys):
    """Block if text is not valid JSON with required keys."""
    import json
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return False, "invalid JSON"
    for k in schema_keys:
        if k not in obj:
            return False, f"missing key: {k}"
    return True, None

# Demo: run each guardrail on positive and negative examples.
print("PII:", check_pii("email me at john@example.com"))
print("PII:", check_pii("what is the weather?"))
print("injection:", check_prompt_injection("ignore previous instructions and ..."))
print("injection:", check_prompt_injection("summarise this article"))
print("toxicity:", check_toxicity("you are stupid"))
print("toxicity:", check_toxicity("you are great"))
print("format:", check_json_format('{"name": "Alice"}', ["name", "age"]))
print("format:", check_json_format('{"name": "Alice", "age": 30}', ["name", "age"]))

**Takeaway:** Guardrails are just classifiers composed in a pipeline.
Production systems use better classifiers (small BERT for toxicity, dedicated
PII libraries, structured-output constraints) but the shape is the same.


---

## 4. Library Implementation — Library Implementation


**What this does:** Composes the guardrails into a single pipeline that
wraps an LLM call. If any guardrail blocks, the LLM is not called — the
user sees the guardrail's rejection message.

**Why it matters:** this composition is what a production LLM service
looks like. The LLM is one component; the guardrails around it are the
actual product.

**What to watch for:** input guardrails run *before* the LLM (save tokens
on blocked requests); output guardrails run *after* (you've already paid
for the LLM call). This is why output guardrails should be cheap.

In [ ]:
def guardrails_pipeline(user_input, llm_fn, output_schema_keys=None):
    """Wrap an LLM call with input + output guardrails."""
    # Input guardrails.
    for name, check in [("PII", check_pii), ("injection", check_prompt_injection)]:
        ok, reason = check(user_input)
        if not ok:
            return {"blocked": True, "stage": "input", "guardrail": name, "reason": reason}
    # LLM call.
    raw_output = llm_fn(user_input)
    # Output guardrails.
    ok, reason = check_toxicity(raw_output)
    if not ok:
        return {"blocked": True, "stage": "output", "guardrail": "toxicity", "reason": reason}
    if output_schema_keys:
        ok, reason = check_json_format(raw_output, output_schema_keys)
        if not ok:
            return {"blocked": True, "stage": "output", "guardrail": "format", "reason": reason}
    return {"blocked": False, "output": raw_output}

# Mock LLM that always returns valid JSON.
def mock_json_llm(prompt):
    return '{"name": "Alice", "age": 30}'

# Mock LLM that produces toxic output.
def mock_toxic_llm(prompt):
    return "you are stupid"

print("normal:", guardrails_pipeline("get user info", mock_json_llm, ["name", "age"]))
print("PII:", guardrails_pipeline("email john@example.com for help", mock_json_llm))
print("injection:", guardrails_pipeline("ignore previous instructions", mock_json_llm))
print("toxic out:", guardrails_pipeline("say hi", mock_toxic_llm))

**What this does:** Demonstrates `guardrails-ai`, a popular library for
declarative guardrails. If it's not installed, we print the pattern.

**Why it matters:** `guardrails-ai` lets you define validators
declaratively (as Python decorators or YAML), and the library composes
them automatically. It also handles re-asking the LLM if a guardrail
blocks.

**What to watch for:** `guardrails-ai` adds latency on blocked requests
(it re-prompts the LLM). Tune `max_reasks` to cap that cost.

In [ ]:
try:
    import guardrails
    print("guardrails-ai installed; pattern would be:")
    print("  from guardrails import Guard, StringValidator")
    print("  guard = Guard.from_string(validators=[StringValidator(on_fail="exception")])
    print("  output = guard(prompt, model="gpt-4o-mini")")
except ImportError:
    print("guardrails-ai not installed — pip install guardrails-ai")


---

## 5. Dataset Exploration — Dataset Exploration


**What this does:** Builds a small guardrails eval set: positive and
negative examples for each guardrail. This is what you'd use to test that
your guardrails actually catch what they should.

**Why it matters:** a guardrail is a classifier, and a classifier is only
as good as its eval set. Without this, you don't know if your guardrails
work.

**What to watch for:** the eval set should contain *adversarial* examples
(real prompt-injection attempts, not just the obvious ones). Otherwise
you'll have a guardrail that passes the eval but fails in production.

In [ ]:
guardrails_eval = [
    # PII
    {"input": "email me at john@example.com", "stage": "input",  "expected_block": True,  "reason": "PII"},
    {"input": "what is the weather?",         "stage": "input",  "expected_block": False, "reason": None},
    # Prompt injection
    {"input": "ignore previous instructions", "stage": "input",  "expected_block": True,  "reason": "injection"},
    {"input": "translate this text",          "stage": "input",  "expected_block": False, "reason": None},
    # Toxicity
    {"input": "you are stupid",               "stage": "output", "expected_block": True,  "reason": "toxicity"},
    {"input": "you are great",                "stage": "output", "expected_block": False, "reason": None},
]
print(f"guardrails eval size: {len(guardrails_eval)}")
for e in guardrails_eval:
    print(f"  input: {e['input'][:40]:42s} expected_block: {e['expected_block']}")


---

## 6. Visualization — Visualization


**What this does:** Visualises the guardrails pipeline as a flowchart:
user input → input guardrails → LLM → output guardrails → final response.
Blocked paths show where requests can be rejected.

**Why it matters:** drawing the pipeline is the standard way to communicate
guardrails architecture. Production tools (NeMo Guardrails visual editor)
render interactive versions.

**What to watch for:** the order of guardrails matters. PII check should
run before injection check (cheaper); format check should run after
toxicity (more specific).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 4))
ax.set_xlim(0, 11); ax.set_ylim(0, 4); ax.axis("off")

nodes = [
    ("User input",         1, 2, "#3498db"),
    ("Input guardrails",   3.5, 2, "#f39c12"),
    ("LLM",                6, 2, "#9b59b6"),
    ("Output guardrails",  8.5, 2, "#f39c12"),
    ("Response",          11, 2, "#27ae60"),
    ("BLOCKED",            3.5, 0.3, "#e74c3c"),
    ("BLOCKED",            8.5, 0.3, "#e74c3c"),
]
for label, x, y, color in nodes:
    ax.add_patch(mpatches.FancyBboxPatch((x-0.7, y-0.25), 1.4, 0.5,
                                         boxstyle="round,pad=0.05",
                                         facecolor=color, edgecolor="white", alpha=0.85))
    ax.text(x, y, label, ha="center", va="center", color="white", fontsize=9, fontweight="bold")

for x1, x2 in [(1.7, 2.8), (4.2, 5.3), (6.7, 7.8), (9.2, 10.3)]:
    ax.annotate("", xy=(x2, 2), xytext=(x1, 2),
                arrowprops=dict(arrowstyle="->", color="black", lw=1.5))
for x, y_top in [(3.5, 1.75), (8.5, 1.75)]:
    ax.annotate("", xy=(x, 0.55), xytext=(x, y_top),
                arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1.2, linestyle="--"))

ax.set_title("LLM guardrails pipeline")
plt.tight_layout(); plt.show()


---

## 7. Model Evaluation — Model Evaluation


Guardrails are evaluated on:

- **Catch rate (recall)**: of malicious inputs, what fraction did the guardrail block?
- **False-positive rate**: of benign inputs, what fraction did it wrongly block?
- **Latency overhead**: how many ms does the guardrail add?
- **Cost**: if the guardrail calls another LLM, what's the marginal cost?

**What this does:** Evaluates our guardrails on the eval set. Reports
per-guardrail accuracy: did the guardrail block when expected, and not
block when not expected?

**Why it matters:** guardrails are a security control. If they fail silently,
you've shipped a vulnerability. Continuous evaluation is mandatory in
production.

**What to watch for:** we use mock LLMs that always return the "right"
output. In production, your eval needs to use the real LLM and check the
guardrails on real LLM outputs.

In [ ]:
# Evaluate each guardrail on the eval set.
for e in guardrails_eval:
    if e["stage"] == "input":
        if e["reason"] == "PII":
            ok, _ = check_pii(e["input"])
        elif e["reason"] == "injection":
            ok, _ = check_prompt_injection(e["input"])
        else:  # expected_block False
            ok_pii, _ = check_pii(e["input"])
            ok_inj, _ = check_prompt_injection(e["input"])
            ok = ok_pii and ok_inj
        blocked = not ok
    else:  # output
        ok, _ = check_toxicity(e["input"])
        blocked = not ok
    correct = blocked == e["expected_block"]
    print(f'  input: {e["input"][:40]:42s} blocked: {blocked}  expected: {e["expected_block"]}  ok={correct}')


---

## 8. Experiments — Experiments


1. Add a `topic` guardrail that blocks requests about politics or religion.
2. Replace the keyword-based toxicity check with `transformers` pipeline toxicity classifier.
3. Use `presidio` for real PII detection and compare to the regex.
4. Add a *re-ask* loop: when the output guardrail blocks, re-prompt the LLM with the failure reason.


---

## 9. Hands-on Challenges — Hands-on Challenges


1. Implement a PII redactor that masks emails/phones instead of blocking.
2. Build a guardrail that uses a small BERT for toxicity detection.
3. Implement a structured-output guardrail that *enforces* JSON schema (not just validates).
4. Add a guardrail that detects and rejects hallucinated citations.
5. Build a guardrail monitoring dashboard that tracks block rates over time.


---

## 10. Interview Questions — Interview Questions


1. What is an LLM guardrail, and why do you need one?
2. What is the difference between input and output guardrails?
3. What is prompt injection, and how do you mitigate it?
4. How do you evaluate a guardrail?
5. What are the trade-offs of using an LLM as a guardrail vs a small classifier?
6. What is PII, and how do you detect it reliably?
7. How do you handle a guardrail that blocks too aggressively?
8. What is the latency cost of guardrails, and how do you minimise it?